# Huấn luyện lại PPO cho GasLeakEnv (notebook chạy độc lập trên Google Colab)

Notebook này **tự chứa toàn bộ mã** (bộ mô phỏng vật lý + môi trường `GasLeakEnv` +
huấn luyện PPO), nên bạn **không cần upload** thư mục dự án. Chỉ cần chọn
`Runtime → Run all`.

Kết quả: file mô hình `ppo_gas_agent.zip`, `vecnormalize.pkl`, file số liệu
`ppo_progress.csv` và **đường cong reward đẹp** `ppo_reward.png` (cùng style với
hình trong khóa luận). Tải về ở cell cuối, rồi chép đè vào
`processing/ml/rl/` và `thesis-latex/img/exp/`.

> Mẹo: để khớp với khóa luận, giữ nguyên `SEED=42`, `TOTAL_TIMESTEPS=400_000`,
> `N_ENVS=4`. Nếu muốn đường cong mượt/đẹp hơn nữa, có thể tăng `TOTAL_TIMESTEPS`.


## 1. Cài đặt thư viện


In [ ]:
!pip -q install "stable-baselines3>=2.4" "gymnasium>=0.29" matplotlib pandas numpy
import stable_baselines3, gymnasium, sys
print('SB3', stable_baselines3.__version__, '| Gymnasium', gymnasium.__version__, '| Python', sys.version.split()[0])

## 2. Tham số huấn luyện
Thay đổi ở đây nếu cần (giữ mặc định để khớp khóa luận).


In [ ]:
SEED = 42
N_ENVS = 4                 # so moi truong song song
EPISODE_SECONDS = 1800     # 30 phut mo phong / episode
TOTAL_TIMESTEPS = 400_000  # tong timesteps (toan cuc)
LEAK_PROB_PER_MIN = 0.05   # xac suat ro ri / phut khi NORMAL

## 3. Bộ mô phỏng vật lý (state machine 4 pha)
Sao chép nguyên văn logic từ `device/simulator/sensor_simulator.py` (phần lõi),
loại bỏ phần MQTT để chạy được độc lập.


In [ ]:
import math, random
from dataclasses import dataclass
from enum import Enum

CRITICAL_PPM = 1000.0

class State(str, Enum):
    NORMAL = "NORMAL"; LEAK_SLOW = "LEAK_SLOW"
    LEAK_FAST = "LEAK_FAST"; VENTILATING = "VENTILATING"

@dataclass
class World:
    gas: float = 60.0; temp: float = 28.0; hum: float = 60.0
    state: State = State.NORMAL; state_age_s: float = 0.0

def _step_normal(w, dt):
    w.gas += (60.0 - w.gas) * 0.1 * dt + random.gauss(0, 3) * dt
    w.gas = max(20.0, min(w.gas, 200.0))
    w.temp += random.gauss(0, 0.05); w.hum += random.gauss(0, 0.1)

def _step_leak_slow(w, dt):
    w.gas += 5.0 * dt + random.gauss(0, 2) * dt
    w.gas = min(w.gas, 1500.0); w.hum += 0.02 * dt

def _step_leak_fast(w, dt):
    w.gas = w.gas * math.exp(0.04 * dt) + 8.0 * dt + random.gauss(0, 3) * dt
    w.gas = min(w.gas, 2000.0); w.hum += 0.05 * dt

def _step_ventilating(w, dt):
    w.gas = max(60.0, w.gas * math.exp(-0.03 * dt) - 0.5 * dt)
    w.temp += random.gauss(0, 0.05); w.hum -= 0.05 * dt

STEP_FN = {State.NORMAL:_step_normal, State.LEAK_SLOW:_step_leak_slow,
           State.LEAK_FAST:_step_leak_fast, State.VENTILATING:_step_ventilating}

def _maybe_transition(w, dt):
    p = LEAK_PROB_PER_MIN / 60.0 * dt
    if w.state == State.NORMAL and random.random() < p:
        w.state = State.LEAK_SLOW if random.random() < 0.7 else State.LEAK_FAST
        w.state_age_s = 0.0; return
    if w.state in (State.LEAK_SLOW, State.LEAK_FAST):
        if w.state_age_s > 180 and random.random() < 0.005:
            w.state = State.VENTILATING; w.state_age_s = 0.0; return
    if w.state == State.VENTILATING and w.gas < 100 and w.state_age_s > 30:
        w.state = State.NORMAL; w.state_age_s = 0.0

def _seconds_to_critical(w):
    if w.gas >= CRITICAL_PPM: return 0
    if w.state == State.LEAK_SLOW: return max(0, int((CRITICAL_PPM - w.gas)/5.0))
    if w.state == State.LEAK_FAST and w.gas > 0:
        return max(0, int(math.log(CRITICAL_PPM / w.gas)/0.04))
    return -1
print("Simulator OK")

## 4. Môi trường `GasLeakEnv` (reward outcome-based v3)
Giống hệt `processing/ml/rl/gas_env.py`.


In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

ACTION_NAMES = ("NO_OP","ALERT_USER","FAN_ON","CLOSE_VALVE")
HORIZON = 300

class GasLeakEnv(gym.Env):
    metadata = {"render_modes": []}
    def __init__(self, episode_seconds=1800, seed=None):
        super().__init__()
        self.episode_seconds = episode_seconds
        self._rng = np.random.default_rng(seed)
        self.observation_space = spaces.Box(0.0, 1.0, shape=(8,), dtype=np.float32)
        self.action_space = spaces.Discrete(4)
        self.world = World(); self.t = 0
        self.fan_on = False; self.valve_closed = False
        self.time_since_action = 0; self._gas_history = []

    def reset(self, *, seed=None, options=None):
        if seed is not None: self._rng = np.random.default_rng(seed)
        self.world = World(); self.t = 0
        self.fan_on = False; self.valve_closed = False
        self.time_since_action = 0; self._gas_history = []
        return self._observe(0.0), {}

    def step(self, action):
        cost = 0.0
        leak_state = self.world.state
        leaking = leak_state in (State.LEAK_SLOW, State.LEAK_FAST)
        if action == 1:
            cost = -3.0 if leak_state == State.NORMAL else 0.0
            self.time_since_action = 0
        elif action == 2:
            if not self.fan_on:
                self.fan_on = True
                if leak_state == State.NORMAL: cost = -15.0
            self.time_since_action = 0
        elif action == 3:
            if not self.valve_closed:
                self.valve_closed = True
                if leaking:
                    self.world.state = State.VENTILATING; self.world.state_age_s = 0.0
                else: cost = -25.0
            self.time_since_action = 0
        else:
            self.time_since_action += 1

        STEP_FN[self.world.state](self.world, 1.0)
        if self.fan_on and self.world.state != State.VENTILATING:
            self.world.gas = max(50.0, self.world.gas * 0.985)
        _maybe_transition(self.world, 1.0)
        self.world.state_age_s += 1.0; self.t += 1
        self._gas_history.append(self.world.gas)

        reward = -0.05 + cost
        if self.world.gas >= CRITICAL_PPM: reward -= 50.0
        if leaking and self.world.gas < CRITICAL_PPM: reward += 0.3
        if self.world.state == State.NORMAL:
            if self.valve_closed: reward -= 0.5
            if self.fan_on: reward -= 0.1
        if self.world.state == State.NORMAL and self.world.state_age_s > 20:
            self.valve_closed = False; self.fan_on = False

        truncated = self.t >= self.episode_seconds
        return (self._observe(self._cheap_forecast()), float(reward), False,
                truncated, {"leak_state": self.world.state.value})

    def _cheap_forecast(self):
        if len(self._gas_history) < 5: return 0.0
        n = min(30, len(self._gas_history))
        slope, _ = np.polyfit(np.arange(n), self._gas_history[-n:], 1)
        pred = self.world.gas + slope * HORIZON
        return float(1.0/(1.0+math.exp(-(pred-CRITICAL_PPM)/200.0)))

    def _slope(self):
        if len(self._gas_history) < 5: return 0.5
        n = min(30, len(self._gas_history))
        slope, _ = np.polyfit(np.arange(n), self._gas_history[-n:], 1)
        return float(np.clip((slope+10.0)/20.0, 0.0, 1.0))

    def _observe(self, p):
        return np.array([min(self.world.gas/2000.0,1.0), min(self.world.temp/60.0,1.0),
                         min(self.world.hum/100.0,1.0), self._slope(),
                         float(np.clip(p,0.0,1.0)), 1.0 if self.fan_on else 0.0,
                         1.0 if self.valve_closed else 0.0,
                         min(self.time_since_action/300.0,1.0)], dtype=np.float32)
print("GasLeakEnv OK")

## 5. Huấn luyện PPO + ghi log reward sạch
Callback ghi lại `(timesteps_toan_cuc, reward_episode)` mỗi khi một episode kết thúc,
phục vụ vẽ đường cong mean ± std.


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize, VecMonitor
from stable_baselines3.common.callbacks import BaseCallback

class RewardLogger(BaseCallback):
    """Ghi lai reward tho moi episode (tu VecMonitor)."""
    def __init__(self):
        super().__init__()
        self.rows = []  # (global_timesteps, episode_reward)
    def _on_step(self):
        for info in self.locals.get("infos", []):
            ep = info.get("episode")
            if ep is not None:
                self.rows.append((int(self.num_timesteps), float(ep["r"])))
        return True

env = make_vec_env(lambda: GasLeakEnv(episode_seconds=EPISODE_SECONDS),
                   n_envs=N_ENVS, seed=SEED)
env = VecMonitor(env)
env = VecNormalize(env, norm_obs=False, norm_reward=True, clip_reward=10.0)

model = PPO("MlpPolicy", env, n_steps=512, batch_size=128, gae_lambda=0.95,
            gamma=0.99, learning_rate=3e-4, ent_coef=0.01, verbose=1, seed=SEED)

logger_cb = RewardLogger()
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=logger_cb)

model.save("ppo_gas_agent.zip")
env.save("vecnormalize.pkl")
print("Da luu mo hinh. So episode ghi duoc:", len(logger_cb.rows))

## 6. Vẽ đường cong reward (mean ± std) và lưu số liệu
Đường cong gộp các episode theo mốc timesteps, vẽ trung bình + dải ±1 độ lệch chuẩn —
tránh các 'gai' từng-episode gây mất thẩm mỹ.


In [ ]:
import pandas as pd, numpy as np
import matplotlib
import matplotlib.pyplot as plt

df = pd.DataFrame(logger_cb.rows, columns=["cum_t", "r"])
df.to_csv("ppo_progress.csv", index=False)

def smooth(y, k=5):
    if len(y) < 3: return y
    k = min(k, len(y)); pad = k//2
    yp = np.pad(y, (pad, pad), mode="edge")
    return np.convolve(yp, np.ones(k)/k, mode="same")[pad:pad+len(y)]

# Gom theo moc timesteps (lam tron de gop cac episode gan nhau)
df["bin"] = (df["cum_t"] // 2000) * 2000
g = df.groupby("bin")["r"].agg(["mean","std"]).reset_index().fillna(0.0)
x = g["bin"].to_numpy(); m = smooth(g["mean"].to_numpy())
up = smooth((g["mean"]+g["std"]).to_numpy()); lo = smooth((g["mean"]-g["std"]).to_numpy())

plt.rcParams["font.family"] = "DejaVu Sans"
fig, ax = plt.subplots(figsize=(10, 5.2))
ax.fill_between(x, lo, up, color="#4a6fa5", alpha=0.18, label="±1 độ lệch chuẩn")
ax.scatter(df["cum_t"], df["r"], s=6, color="#9bb0cc", alpha=0.22, linewidths=0,
           label="reward từng episode")
ax.plot(x, m, color="#1f3a5f", lw=2.6, label="reward trung bình (đã làm mượt)")
final = m[-1]
ax.axhline(final, color="#5ba26c", ls="--", lw=1.2)
ax.annotate("reward hội tụ ≈ %.0f" % final, xy=(x[-1]*0.95, final),
            xytext=(x[-1]*0.5, ax.get_ylim()[0]*0.25), color="#2f6b46",
            weight="bold", fontsize=11,
            arrowprops=dict(arrowstyle="->", color="#5ba26c", lw=1.3))
ax.set_title("Đường cong reward huấn luyện PPO trên GasLeakEnv (%d môi trường)" % N_ENVS,
             fontsize=12.5, weight="bold")
ax.set_xlabel("Tổng số timesteps huấn luyện"); ax.set_ylabel("Reward mỗi episode")
ax.grid(True, ls=":", alpha=0.45); ax.legend(loc="lower right", fontsize=9.5)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: ("%.0fk"%(v/1000)) if v else "0"))
fig.tight_layout(); fig.savefig("ppo_reward.png", dpi=170, bbox_inches="tight", facecolor="white")
plt.show()
print("Da luu ppo_reward.png va ppo_progress.csv")

## 7. (Tuỳ chọn) Đánh giá nhanh agent vừa huấn luyện
Chạy thử 1 episode để xem agent có giữ khí dưới ngưỡng không.


In [ ]:
eval_env = GasLeakEnv(episode_seconds=1800, seed=123)
obs, _ = eval_env.reset()
peak = 0.0; total = 0.0; breaches = 0
for _ in range(1800):
    a, _ = model.predict(obs, deterministic=True)
    obs, r, term, trunc, info = eval_env.step(int(a))
    peak = max(peak, eval_env.world.gas); total += r
    if eval_env.world.gas >= CRITICAL_PPM: breaches += 1
    if term or trunc: break
print(f"Peak gas = {peak:.0f} ppm | so buoc vuot nguong = {breaches} | tong reward = {total:.0f}")

## 8. Tải các file kết quả về máy
Tải `ppo_gas_agent.zip`, `vecnormalize.pkl` → chép vào `processing/ml/rl/`.
Tải `ppo_reward.png`, `ppo_progress.csv` → chép vào `thesis-latex/img/exp/`.


In [ ]:
from google.colab import files
for f in ["ppo_gas_agent.zip", "vecnormalize.pkl", "ppo_reward.png", "ppo_progress.csv"]:
    try: files.download(f)
    except Exception as e: print("Bo qua", f, e)